In [ ]:
# [1] 설치 + 설정 + 업비트 원화마켓 시간봉 불러오기 (처음엔 몇 분 걸려요)
!pip install -q ccxt
import os, time, random, pickle
import ccxt
import pandas as pd
import numpy as np

SYMS = ["BTC/KRW","ETH/KRW","XRP/KRW","SOL/KRW","ADA/KRW","DOGE/KRW",
        "LINK/KRW","AVAX/KRW","DOT/KRW","BCH/KRW"]
SINCE = "2022-01-01"
SPLIT = pd.Timestamp("2024-07-01")   # 기간을 반으로 나눠 확인
FEE = 0.001   # 편도 0.1% 가정 (업비트 수수료 약 0.05% + 슬리피지)
CACHE = "/content/upbit_1h.pkl"

ex = ccxt.upbit()


def fetch_all(sym):
    since = int(pd.Timestamp(SINCE).timestamp() * 1000) - 9 * 3600 * 1000
    out = []
    while True:
        try:
            rows = ex.fetch_ohlcv(sym, "1h", since=since, limit=200)
        except Exception as e:
            print("오류", sym, str(e)[:80])
            break
        if not rows:
            break
        out += rows
        nxt = rows[-1][0] + 1
        if nxt <= since:
            break
        since = nxt
        if rows[-1][0] > time.time() * 1000 - 2 * 3600 * 1000:
            break
        time.sleep(ex.rateLimit / 1000)
    if not out:
        return None
    df = pd.DataFrame(out, columns=["ts", "Open", "High", "Low", "Close", "Volume"])
    df = df.drop_duplicates("ts").sort_values("ts")
    df.index = pd.to_datetime(df.ts, unit="ms") + pd.Timedelta(hours=9)  # 한국시간
    return df[["Open", "High", "Low", "Close", "Volume"]]


if os.path.exists(CACHE):
    H1 = pickle.load(open(CACHE, "rb"))
else:
    H1 = {}
    for s in SYMS:
        d = fetch_all(s)
        if d is not None and len(d) > 2000:
            H1[s] = d
            print(s, len(d), "개 캔들")
        else:
            print("건너뜀:", s)
    pickle.dump(H1, open(CACHE, "wb"))


def resample(d, tf):
    if tf == "1h":
        return d
    rule, off = ("4h", "1h") if tf == "4h" else ("24h", "9h")  # 일봉은 한국시간 오전 9시 기준(업비트와 동일)
    o = d.resample(rule, offset=off)
    r = pd.DataFrame({"Open": o.Open.first(), "High": o.High.max(),
                      "Low": o.Low.min(), "Close": o.Close.last(),
                      "Volume": o.Volume.sum()}).dropna()
    return r[r.Volume > 0]


TFS = {"1h": 24, "4h": 6, "1d": 1}   # 하루에 몇 개 봉인지
DATA = {tf: {s: resample(d, tf) for s, d in H1.items()} for tf in TFS}
print(f"{len(H1)}개 코인 준비 완료:", list(H1))

In [ ]:
# [2] 핵심 함수: 돌파 신호/체결 + 계좌 시뮬레이션
def get_trades(d, win=60, vol=0, stop_below=0.04, atr_mult=3.0, maxrisk=0.15):
    O, H, L, C = (d[c].values for c in ["Open", "High", "Low", "Close"])
    Cs, Hs, Ls, Vs = d.Close, d.High, d.Low, d.Volume
    n = len(d)
    resist = Hs.rolling(win).max().shift(1)
    broke = (Cs > resist) & (Cs.shift(1) <= resist)
    if vol:
        broke &= Vs > Vs.rolling(20).mean().shift(1) * vol
    lvl = resist.shift(1)
    conf = broke.shift(1, fill_value=False) & (Ls >= lvl * 0.99) & (Cs > lvl)
    nxt = Hs.rolling(win * 4).max().shift(1).values   # 1차 목표: 돌파창의 4배 기간 고점
    pc = Cs.shift(1)
    tr = pd.concat([Hs - Ls, (Hs - pc).abs(), (Ls - pc).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean().values
    lv = lvl.values
    out, free = [], 0
    for i in np.where(conf.values)[0]:
        j = i + 1
        if j >= n or j < free:
            continue
        entry, stop = O[j], lv[i] * (1 - stop_below)
        if not stop < entry:
            continue
        risk = (entry - stop) / entry
        if not 0.003 <= risk <= maxrisk:
            continue
        tgt = nxt[i]
        if np.isnan(tgt) or tgt < entry * (1 + risk):
            tgt = entry * (1 + 2 * risk)
        half, hi, tstop, part = False, entry, 0.0, 0.0
        xk, xp = n - 1, C[-1]
        for k in range(j, n):
            cur = max(tstop, entry) if half else stop
            if k > j and O[k] <= cur:
                xk, xp = k, O[k]
                break
            if L[k] <= cur:
                xk, xp = k, cur
                break
            if (not half) and H[k] >= tgt:
                half, part = True, tgt / entry - 1
            hi = max(hi, H[k])
            if half:
                tstop = max(tstop, hi - atr_mult * atr[k])
        ret = (0.5 * part + 0.5 * (xp / entry - 1)) if half else (xp / entry - 1)
        ret -= 2 * FEE
        free = xk + 1
        out.append(dict(entry=d.index[j], exit=d.index[xk], ret=ret, risk=risk))
    return out


def simulate(trades, max_pos=5, risk_pct=0.01, max_alloc=0.25, init=1_000_000):
    cash, openp, curve, skipped = init, [], [], 0

    def settle(dt):
        nonlocal cash
        for p in sorted(openp, key=lambda p: p["exit"]):
            if p["exit"] <= dt:
                cash += p["alloc"] * (1 + p["ret"])
                openp.remove(p)
                curve.append((p["exit"], cash + sum(q["alloc"] for q in openp)))

    for t in trades:
        settle(t["entry"])
        eq = cash + sum(p["alloc"] for p in openp)
        if len(openp) >= max_pos:
            skipped += 1
            continue
        alloc = min(eq * risk_pct / t["risk"], eq * max_alloc, cash)
        if alloc < eq * 0.02:
            skipped += 1
            continue
        cash -= alloc
        openp.append(dict(t, alloc=alloc))
    settle(pd.Timestamp("2100-01-01"))
    s = pd.Series([v for _, v in curve],
                  index=pd.DatetimeIndex([x for x, _ in curve]))
    s = s.groupby(level=0).last()
    s = pd.concat([pd.Series([init], index=[trades[0]["entry"]]), s])
    yrs = max((s.index[-1] - s.index[0]).days / 365.25, 0.1)
    hold = np.mean([(t["exit"] - t["entry"]).total_seconds() / 3600 for t in trades])
    return dict(final=s.iloc[-1], cagr=(s.iloc[-1] / init) ** (1 / yrs) - 1,
                mdd=(s / s.cummax() - 1).min(), n=len(trades) - skipped,
                skip=skipped, hold=hold, curve=s)


def build(tf, win_days, stop_below, pool=None, **kw):
    tr = []
    for sym, d in DATA[tf].items():
        if pool is None or sym in pool:
            tr += get_trades(d, win=win_days * TFS[tf], stop_below=stop_below, **kw)
    return sorted(tr, key=lambda x: x["entry"])


def go(tr, **kw):
    return simulate(tr, **kw) if len(tr) >= 30 else None


print("함수 준비 완료")

In [ ]:
# [3] 시간봉 비교: 1시간 / 4시간 / 일봉 x 돌파기간 3/7/20일 (비용 1배 vs 3배)
STOPS = {"1h": 0.02, "4h": 0.03, "1d": 0.05}
print(f"{'봉':>3s}{'돌파일':>5s}{'연평균':>8s}{'MDD':>8s}{'매매':>6s}{'평균보유(시간)':>13s}{'비용3배':>9s}")
for tf in ["1h", "4h", "1d"]:
    for wd in [3, 7, 20]:
        FEE = 0.001
        r1 = go(build(tf, wd, STOPS[tf]))
        FEE = 0.003
        r3 = go(build(tf, wd, STOPS[tf]))
        FEE = 0.001
        if r1 is None:
            print(f"{tf:>3s}{wd:>5d}  신호 부족")
            continue
        c3 = "  -  " if r3 is None else f"{r3['cagr']:.1%}"
        print(f"{tf:>3s}{wd:>5d}{r1['cagr']:>8.1%}{r1['mdd']:>8.1%}{r1['n']:>6d}"
              f"{r1['hold']:>13.0f}{c3:>9s}")

In [ ]:
# [4] 시간봉별 견고성: 기간 분할 / 코인 절반 제거 (돌파 7일 기준)
for tf in ["1h", "4h", "1d"]:
    print(f"\n===== {tf} (돌파 7일, 손절 {STOPS[tf]:.0%}) =====")
    base = build(tf, 7, STOPS[tf])
    for name, sub in [("전반", [x for x in base if x["entry"] < SPLIT]),
                      ("후반", [x for x in base if x["entry"] >= SPLIT])]:
        rr = go(sub)
        if rr is None:
            print(f"  {name}: 신호 부족({len(sub)}개)")
        else:
            print(f"  {name}: 연평균 {rr['cagr']:.1%} | MDD {rr['mdd']:.1%} | {rr['n']}회")
    res = []
    for seed in range(6):
        random.seed(seed)
        keep = set(random.sample(list(DATA[tf]), len(DATA[tf]) // 2))
        rr = go(build(tf, 7, STOPS[tf], pool=keep))
        if rr is not None:
            res.append(rr["cagr"])
    if res:
        print(f"  코인 절반 제거 {len(res)}회: 플러스 {sum(x > 0 for x in res)}/{len(res)}"
              f" | 중앙값 {np.median(res):.1%}")